# Data Preprocessing - Construction PPE Dataset
Cleaning, normalization, dan preparation untuk training

## 1. Import Libraries

In [1]:
import os
import shutil
import json
from pathlib import Path
from collections import Counter
import numpy as np
import cv2
import yaml
from tqdm import tqdm

dataset_root = '../data'
output_root = '../data/processed'

print(f'Input: {dataset_root}')
print(f'Output: {output_root}')

Input: ../data
Output: ../data/processed


## 2. Define Class Mapping and Cleaning Rules

In [2]:
class_mapping = {
    'helmet': 0,
    'gloves': 1,
    'vest': 2,
    'boots': 3,
    'goggles': 4,
    'person': 5
}

original_class_names = {
    0: 'helmet',
    1: 'gloves',
    2: 'vest',
    3: 'boots',
    4: 'goggles',
    5: 'none',
    6: 'Person',
    7: 'no_helmet',
    8: 'no_goggle',
    9: 'no_gloves',
    10: 'no_boots'
}

classes_to_keep = {0, 1, 2, 3, 4, 6}
classes_to_remove = {5, 7, 8, 9, 10}

print('Classes to KEEP:')
for cid in sorted(classes_to_keep):
    print(f'  {cid}: {original_class_names[cid]}')

print('\nClasses to REMOVE:')
for cid in sorted(classes_to_remove):
    print(f'  {cid}: {original_class_names[cid]}')

print('\nNew class mapping:')
for name, new_id in class_mapping.items():
    print(f'  {new_id}: {name}')

Classes to KEEP:
  0: helmet
  1: gloves
  2: vest
  3: boots
  4: goggles
  6: Person

Classes to REMOVE:
  5: none
  7: no_helmet
  8: no_goggle
  9: no_gloves
  10: no_boots

New class mapping:
  0: helmet
  1: gloves
  2: vest
  3: boots
  4: goggles
  5: person


## 3. Create Output Directory Structure

In [3]:
def create_directory_structure(root_path):
    splits = ['train', 'val', 'test']
    
    for split in splits:
        images_dir = os.path.join(root_path, 'images', split)
        labels_dir = os.path.join(root_path, 'labels', split)
        
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)
    
    print(f'Created directory structure at {root_path}')

create_directory_structure(output_root)

Created directory structure at ../data/processed


## 4. Process Annotations and Images

In [4]:
def remap_class_id(original_id):
    original_name = original_class_names.get(original_id)
    if original_name:
        original_name_lower = original_name.lower()
        return class_mapping.get(original_name_lower)
    return None

def process_annotation_file(input_path, output_path):
    cleaned_lines = []
    
    if not os.path.exists(input_path):
        return True
    
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    original_class_id = int(parts[0])
                    
                    if original_class_id in classes_to_remove:
                        continue
                    
                    new_class_id = remap_class_id(original_class_id)
                    if new_class_id is not None:
                        new_line = f"{new_class_id} {parts[1]} {parts[2]} {parts[3]} {parts[4]}\n"
                        cleaned_lines.append(new_line)
        
        if cleaned_lines:
            with open(output_path, 'w', encoding='utf-8') as f:
                f.writelines(cleaned_lines)
            return True
        else:
            return False
    except Exception as e:
        return False

print('Annotation processing function defined')

Annotation processing function defined


## 5. Copy and Process Dataset

In [5]:
splits = ['train', 'val', 'test']
stats = {split: {'kept': 0, 'removed': 0} for split in splits}

for split in splits:
    print(f'\nProcessing {split.upper()}...')
    
    input_images_dir = os.path.join(dataset_root, 'images', split)
    input_labels_dir = os.path.join(dataset_root, 'labels', split)
    output_images_dir = os.path.join(output_root, 'images', split)
    output_labels_dir = os.path.join(output_root, 'labels', split)
    
    if not os.path.exists(input_images_dir):
        print(f'  Input directory not found: {input_images_dir}')
        continue
    
    image_files = [f for f in os.listdir(input_images_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    for img_file in tqdm(image_files, desc=f'Processing {split}'):
        input_img_path = os.path.join(input_images_dir, img_file)
        output_img_path = os.path.join(output_images_dir, img_file)
        
        label_file = img_file.rsplit('.', 1)[0] + '.txt'
        input_label_path = os.path.join(input_labels_dir, label_file)
        output_label_path = os.path.join(output_labels_dir, label_file)
        
        keep_image = process_annotation_file(input_label_path, output_label_path)
        
        if keep_image:
            shutil.copy2(input_img_path, output_img_path)
            stats[split]['kept'] += 1
        else:
            stats[split]['removed'] += 1
    
    print(f'  Kept: {stats[split]["kept"]} images')
    print(f'  Removed: {stats[split]["removed"]} images')


Processing TRAIN...


Processing train: 100%|██████████| 1132/1132 [00:04<00:00, 264.74it/s]


  Kept: 1128 images
  Removed: 4 images

Processing VAL...


Processing val: 100%|██████████| 143/143 [00:00<00:00, 330.19it/s]


  Kept: 143 images
  Removed: 0 images

Processing TEST...


Processing test: 100%|██████████| 141/141 [00:00<00:00, 309.96it/s]

  Kept: 141 images
  Removed: 0 images


## 6. Create data.yaml for YOLO

In [6]:
yaml_config = {
    'path': os.path.abspath(output_root),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(class_mapping),
    'names': {v: k for k, v in class_mapping.items()}
}

yaml_path = os.path.join(output_root, 'data.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_config, f, default_flow_style=False, sort_keys=False)

print(f'Created data.yaml at {yaml_path}')
print('\nYAML content:')
with open(yaml_path, 'r', encoding='utf-8') as f:
    print(f.read())

Created data.yaml at ../data/processed\data.yaml

YAML content:
path: c:\Users\Rafif Fikri\OneDrive\Documents\PROJECT SERIUS\PPE_Risk\data\processed
train: images/train
val: images/val
test: images/test
nc: 6
names:
  0: helmet
  1: gloves
  2: vest
  3: boots
  4: goggles
  5: person



## 7. Validation Check

In [7]:
def validate_processed_dataset(root_path):
    issues = []
    
    for split in ['train', 'val', 'test']:
        images_dir = os.path.join(root_path, 'images', split)
        labels_dir = os.path.join(root_path, 'labels', split)
        
        if not os.path.exists(images_dir):
            issues.append(f'Missing images directory for {split}')
            continue
        
        image_files = [f for f in os.listdir(images_dir) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        for img_file in image_files:
            label_file = img_file.rsplit('.', 1)[0] + '.txt'
            label_path = os.path.join(labels_dir, label_file)
            
            if not os.path.exists(label_path):
                issues.append(f'{split}: Missing label for {img_file}')
    
    return issues

issues = validate_processed_dataset(output_root)
if issues:
    print('Validation issues found:')
    for issue in issues:
        print(f'  - {issue}')
else:
    print('Validation passed. Dataset is ready for training.')

Validation passed. Dataset is ready for training.


## 8. Summary Statistics

In [8]:
print('Processing Summary:')
print('=' * 50)
for split in splits:
    print(f'{split.upper()}:')
    print(f'  Images kept:    {stats[split]["kept"]}')
    print(f'  Images removed: {stats[split]["removed"]}')
    print()

print('\nClass Mapping Summary:')
print('=' * 50)
for class_id, class_name in class_mapping.items():
    print(f'{class_id}: {class_name}')

print('\nOutput Location:')
print(f'{output_root}')

Processing Summary:
TRAIN:
  Images kept:    1128
  Images removed: 4

VAL:
  Images kept:    143
  Images removed: 0

TEST:
  Images kept:    141
  Images removed: 0


Class Mapping Summary:
helmet: 0
gloves: 1
vest: 2
boots: 3
goggles: 4
person: 5

Output Location:
../data/processed
